# DWD-Wetterdaten für Berlin, 2020 bis aktuell

Dieses Notebook lädt ausschließlich tägliche DWD-Daten der Station **Berlin-Dahlem (FU), Stations-ID 00403**, und speichert sie als CSV. Die gewünschte Auswertung folgt später; hier prüfen wir zunächst nur Datenquelle, Zeitraum, Spalten und Vollständigkeit.

Die Archivprüfung am 02.11.2026 fand Daten vom **01.01.2020 bis 01.11.2026**. Die fünf Variablen sind vorhanden. Einzelne Tage oder Messwerte fehlen; sie bleiben in der CSV leer statt durch erfundene Werte ersetzt zu werden.

**CSV-Datei:** `dwd_berlin_dahlem_tagesdaten_2020_bis_aktuell.csv`

## Datenquelle und Variablen

Für die tägliche Klimavariable nutzt der DWD diese Spalten und Einheiten:

- `TNK`: Tagesminimum der Lufttemperatur in °C
- `TXK`: Tagesmaximum der Lufttemperatur in °C
- `TMK`: Tagesmitteltemperatur in °C
- `RSK`: tägliche Niederschlagshöhe in mm
- `SDK`: tägliche Sonnenscheindauer in Stunden

Die historischen und aktuellen Archive überlappen teilweise. Beim Zusammenführen erhält ein Eintrag aus dem aktuellen Archiv Vorrang.

In [1]:
import io
import zipfile
from urllib.request import urlopen

import pandas as pd

DWD_BASIS = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/"
DWD_ARCHIVE = [
    "historical/tageswerte_KL_00403_19500101_20251231_hist.zip",
    "recent/tageswerte_KL_00403_akt.zip",
]
CSV_DATEI = "dwd_berlin_dahlem_tagesdaten_2020_bis_aktuell.csv"

archiv_tabellen = []
for archiv_pfad in DWD_ARCHIVE:
    with urlopen(DWD_BASIS + archiv_pfad, timeout=60) as antwort:
        zip_daten = antwort.read()

    with zipfile.ZipFile(io.BytesIO(zip_daten)) as archiv:
        produktdatei = next(
            name
            for name in archiv.namelist()
            if name.startswith("produkt_klima_tag_") and name.endswith(".txt")
        )
        with archiv.open(produktdatei) as csv_daten:
            tabelle = pd.read_csv(
                csv_daten,
                sep=";",
                skipinitialspace=True,
                na_values=[-999, "-999"],
                encoding="cp1252",
                low_memory=False,
            )
    tabelle.columns = tabelle.columns.str.strip()
    archiv_tabellen.append(tabelle)

# Bei überlappenden Tagen gewinnt das aktuelle Archiv.
dwd_roh = pd.concat(archiv_tabellen, ignore_index=True)
dwd_roh["MESS_DATUM"] = pd.to_datetime(
    dwd_roh["MESS_DATUM"].astype("Int64").astype(str),
    format="%Y%m%d",
)
dwd_roh = dwd_roh.drop_duplicates(subset="MESS_DATUM", keep="last")

letzter_dwd_tag = min(pd.Timestamp.today().normalize(), dwd_roh["MESS_DATUM"].max())
start = pd.Timestamp("2020-01-01")
spalten = {
    "MESS_DATUM": "datum",
    "TNK": "temp_min_c",
    "TXK": "temp_max_c",
    "TMK": "temp_avg_c",
    "RSK": "niederschlagsmenge_mm",
    "SDK": "sonnenstunden_h",
}

dwd_csv = dwd_roh.loc[
    dwd_roh["MESS_DATUM"].between(start, letzter_dwd_tag),
    list(spalten),
].rename(columns=spalten)

# Fehlende Kalendertage bleiben als Zeilen mit leeren Messwerten sichtbar.
dwd_csv = (
    dwd_csv.set_index("datum")
    .reindex(pd.date_range(start, letzter_dwd_tag, freq="D", name="datum"))
    .reset_index()
)
dwd_csv.to_csv(CSV_DATEI, index=False, date_format="%Y-%m-%d", na_rep="")

print(f"Gespeichert: {CSV_DATEI}")
print(f"Station: Berlin-Dahlem (FU), 00403")
print(f"Zeitraum: {dwd_csv['datum'].min().date()} bis {dwd_csv['datum'].max().date()}")
print(f"Kalendertage in CSV: {len(dwd_csv)}")
print("Gültige Werte je Variable:\n", dwd_csv.notna().sum())
dwd_csv.head()

Gespeichert: dwd_berlin_dahlem_tagesdaten_2020_bis_aktuell.csv
Station: Berlin-Dahlem (FU), 00403
Zeitraum: 2020-01-01 bis 2026-11-01
Kalendertage in CSV: 2497
Gültige Werte je Variable:
 datum                    2497
temp_min_c               2481
temp_max_c               2481
temp_avg_c               2481
niederschlagsmenge_mm    2482
sonnenstunden_h          2481
dtype: int64


,datum,temp_min_c,temp_max_c,temp_avg_c,niederschlagsmenge_mm,sonnenstunden_h
0,2020-01-01,-2.8,5.0,1.8,0.0,2.9
1,2020-01-02,-4.1,6.2,-0.3,0.0,7.5
2,2020-01-03,0.8,6.9,4.3,4.4,0.0
3,2020-01-04,1.7,5.6,4.1,3.0,0.0
4,2020-01-05,-1.4,3.6,1.4,0.0,2.1


## CSV-Datei und Verfügbarkeit prüfen

Diese Zelle liest die gespeicherte CSV erneut ein, prüft die erwarteten Spalten und zeigt gültige sowie fehlende Tageswerte. Sie enthält bewusst noch keine statistische Wetterauswertung.

In [2]:
daten = pd.read_csv(CSV_DATEI, parse_dates=["datum"])
benoetigte_spalten = [
    "datum",
    "temp_min_c",
    "temp_max_c",
    "temp_avg_c",
    "niederschlagsmenge_mm",
    "sonnenstunden_h",
]
fehlende_spalten = set(benoetigte_spalten) - set(daten.columns)
assert not fehlende_spalten, f"Fehlende Spalten: {fehlende_spalten}"
assert daten["datum"].is_monotonic_increasing
assert daten["datum"].is_unique
assert len(daten) == (daten["datum"].max() - daten["datum"].min()).days + 1

print(f"Station: Berlin-Dahlem (FU), 00403")
print(f"CSV-Zeitraum: {daten['datum'].min().date()} bis {daten['datum'].max().date()}")
print(f"Kalendertage: {len(daten)}")

verfuegbarkeit = pd.DataFrame(
    {
        "gueltige_tage": daten[benoetigte_spalten[1:]].notna().sum(),
        "fehlende_tage": daten[benoetigte_spalten[1:]].isna().sum(),
        "verfuegbar_prozent": (
            daten[benoetigte_spalten[1:]].notna().mean() * 100
        ).round(2),
    }
)
verfuegbarkeit

Station: Berlin-Dahlem (FU), 00403
CSV-Zeitraum: 2020-01-01 bis 2026-11-01
Kalendertage: 2497


,gueltige_tage,fehlende_tage,verfuegbar_prozent
temp_min_c,2481,16,99.36
temp_max_c,2481,16,99.36
temp_avg_c,2481,16,99.36
niederschlagsmenge_mm,2482,15,99.40
sonnenstunden_h,2481,16,99.36
